# ML Assignment 2


## Dataset loading

We will load the Breast Cancer Wisconsin Diagnostic dataset from scikit-learn, which mirrors the UCI dataset and avoids manual download steps.


In [1]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df.head()


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


## Dataset inspection

Start with the assignment checks: dataset size, feature list, target meaning, class balance, and missing-value inspection.


In [2]:
X = data.data.copy()
y = data.target.copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target names: {list(data.target_names)}")
print(f"Total feature count: {len(data.feature_names)}")

feature_summary = pd.DataFrame({
    "feature_name": data.feature_names,
    "dtype": X.dtypes.astype(str).values,
})
feature_summary.head(30)


Feature matrix shape: (569, 30)
Target shape: (569,)
Target names: [np.str_('malignant'), np.str_('benign')]
Total feature count: 30


,feature_name,dtype
0,mean radius,float64
1,mean texture,float64
2,mean perimeter,float64
3,mean area,float64
4,mean smoothness,float64
5,mean compactness,float64
6,mean concavity,float64
7,mean concave points,float64
8,mean symmetry,float64
9,mean fractal dimension,float64


In [3]:
class_balance = y.value_counts().rename(index={0: "malignant", 1: "benign"}).to_frame("count")
class_balance["percentage"] = (class_balance["count"] / len(y) * 100).round(2)
class_balance


,count,percentage
target,,
benign,357,62.74
malignant,212,37.26


In [4]:
missing_values = X.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]


Series([], dtype: int64)

## Preprocessing strategy

A single identical preprocessing step is not ideal for all five models.

- **Logistic Regression** and **KNN** benefit from feature scaling.
- **Gaussian Naive Bayes** also works well with numeric features and can use scaled inputs consistently.
- **Decision Tree** and **Random Forest** do not require scaling because tree splits are scale-invariant.

Since this dataset is already numeric and has no categorical columns, the practical plan is:

1. No encoding needed.
2. No imputation needed if missing values remain absent.
3. Use **StandardScaler** inside pipelines for Logistic Regression, KNN, and Gaussian Naive Bayes.
4. Use raw numeric features for Decision Tree and Random Forest.

This keeps preprocessing fair and model-appropriate while still maintaining a clean common workflow.


## Train/test split

Use a stratified split so the benign/malignant ratio stays consistent in both sets.


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

split_balance = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "test_count": y_test.value_counts().sort_index(),
})
split_balance.index = split_balance.index.map({0: "malignant", 1: "benign"})
split_balance


X_train shape: (455, 30)
X_test shape: (114, 30)
y_train shape: (455,)
y_test shape: (114,)


,train_count,test_count
target,,
malignant,170,42
benign,285,72


## Model pipelines and metrics table

Build one pipeline per model, fit on the training set, and compare all assignment metrics on the held-out test set.


In [6]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

positive_label = 0  # malignant

pipelines = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(random_state=42)),
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB()),
    ]),
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
}

trained_models = {}
results = []

for model_name, pipeline in pipelines.items():
    fitted_pipeline = clone(pipeline)
    fitted_pipeline.fit(X_train, y_train)
    trained_models[model_name] = fitted_pipeline

    y_pred = fitted_pipeline.predict(X_test)
    malignant_class_index = list(fitted_pipeline.named_steps["model"].classes_).index(positive_label)
    malignant_proba = fitted_pipeline.predict_proba(X_test)[:, malignant_class_index]

    results.append({
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score((y_test == positive_label).astype(int), malignant_proba),
        "Precision": precision_score(y_test, y_pred, pos_label=positive_label),
        "Recall": recall_score(y_test, y_pred, pos_label=positive_label),
        "F1": f1_score(y_test, y_pred, pos_label=positive_label),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

comparison_table = pd.DataFrame(results).sort_values(by=["F1", "AUC"], ascending=False)
comparison_table = comparison_table.reset_index(drop=True)
comparison_table.style.format({
    "Accuracy": "{:.4f}",
    "AUC": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "MCC": "{:.4f}",
})


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9825,0.9954,0.9762,0.9762,0.9762,0.9623
1,Random Forest,0.9561,0.9931,0.9512,0.9286,0.9398,0.9054
2,KNN,0.9561,0.9788,0.9512,0.9286,0.9398,0.9054
3,Gaussian Naive Bayes,0.9298,0.9868,0.9048,0.9048,0.9048,0.8492
4,Decision Tree,0.9123,0.9157,0.8478,0.9286,0.8864,0.8174
